In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
from pathlib import Path

data_path = '/content/drive/My Drive/sql_data_csv'
print(os.listdir(data_path))

['periods.csv', 'marketing_spend.csv', 'orders.csv']


In [ ]:
import pandas as pd
base_path = Path(data_path)
marketing = pd.read_csv(base_path/'marketing_spend.csv')
marketing.head()

,spend_id,period_id,month_start,channel_id,purpose,amount
0,M00001,FY2024_25,2024-04-01,APP,Acquisition,960718
1,M00002,FY2024_25,2024-04-01,APP,Retention,535845
2,M00003,FY2024_25,2024-04-01,WEB,Acquisition,1067938
3,M00004,FY2024_25,2024-04-01,WEB,Retention,601378
4,M00005,FY2024_25,2024-04-01,STORE,Acquisition,1389923


In [ ]:
import pandas as pd
base_path = Path(data_path)
orders = pd.read_csv(base_path/'orders.csv')
orders.head()

,order_id,customer_id,period_id,order_date,channel_id,status,gross_amount,discount_amount,refund_amount
0,O0000025,C000007,FY2024_25,2024-04-01,STORE,COMPLETED,163890,8560,0
1,O0000226,C000116,FY2024_25,2024-04-01,STORE,COMPLETED,83014,7568,0
2,O0000953,C000499,FY2024_25,2024-04-01,STORE,COMPLETED,198174,18560,0
3,O0001380,C000452,FY2024_25,2024-04-01,STORE,COMPLETED,121126,8176,0
4,O0001408,C000184,FY2024_25,2024-04-01,STORE,COMPLETED,31871,939,0


##**Step 1:** Which orders belong in revenue?

In [ ]:
orders = orders[orders['status']=='COMPLETED']
orders[('net_revenue')] = orders['gross_amount'] - orders['discount_amount'] - orders['refund_amount']
orders.head()

,order_id,customer_id,period_id,order_date,channel_id,status,gross_amount,discount_amount,refund_amount,net_revenue
0,O0000025,C000007,FY2024_25,2024-04-01,STORE,COMPLETED,163890,8560,0,155330
1,O0000226,C000116,FY2024_25,2024-04-01,STORE,COMPLETED,83014,7568,0,75446
2,O0000953,C000499,FY2024_25,2024-04-01,STORE,COMPLETED,198174,18560,0,179614
3,O0001380,C000452,FY2024_25,2024-04-01,STORE,COMPLETED,121126,8176,0,112950
4,O0001408,C000184,FY2024_25,2024-04-01,STORE,COMPLETED,31871,939,0,30932


In [ ]:
# Find the total marketing spent by period

marketing.groupby('period_id')['amount'].sum()

,amount
period_id,
FY2024_25,70000000
FY2025_26,95200000


In [ ]:
orders.groupby('period_id')['net_revenue'].sum()

,net_revenue
period_id,
FY2024_25,1000000000
FY2025_26,1040000000


In [ ]:
orders.groupby('period_id')['customer_id'].nunique()

,net_revenue
period_id,
FY2024_25,1000000000
FY2025_26,1040000000


In [15]:
summary = pd.DataFrame({"marketing_spend":marketing.groupby('period_id')['amount'].sum(),
 'revenue':orders.groupby('period_id')['net_revenue'].sum(),
 'customers': orders.groupby('period_id')['customer_id'].nunique()
 })

In [16]:
summary['marketing_spend'] = summary['marketing_spend']/10000000
summary['revenue'] = summary['revenue']/10000000
summary

,marketing_spend,revenue,customers
period_id,,,
FY2024_25,7.00,100.0,10000
FY2025_26,9.52,104.0,13000


In [17]:
current_year = summary.loc['FY2025_26']
current_year

,FY2025_26
marketing_spend,9.52
revenue,104.00
customers,13000.00


In [18]:
prev_year = summary.loc['FY2024_25']
prev_year

,FY2024_25
marketing_spend,7.0
revenue,100.0
customers,10000.0


In [19]:
(current_year - prev_year)/prev_year*100

,0
marketing_spend,36.0
revenue,4.0
customers,30.0


In [1]:
summary

NameError: name 'summary' is not defined

### Marketing spend as a part of Revenue - Why? Because we need to be aware of constraints

In [ ]:
marketing_percentage = summary['marketing_spend']/summary['revenue']*100
marketing_percentage.round(2)

In [ ]:
marketing

,spend_id,period_id,month_start,channel_id,purpose,amount
0,M00001,FY2024_25,2024-04-01,APP,Acquisition,960718
1,M00002,FY2024_25,2024-04-01,APP,Retention,535845
2,M00003,FY2024_25,2024-04-01,WEB,Acquisition,1067938
3,M00004,FY2024_25,2024-04-01,WEB,Retention,601378
4,M00005,FY2024_25,2024-04-01,STORE,Acquisition,1389923
...,...,...,...,...,...,...
139,M00140,FY2025_26,2026-03-01,APP,Retention,1051187
140,M00141,FY2025_26,2026-03-01,WEB,Acquisition,1758647
141,M00142,FY2025_26,2026-03-01,WEB,Retention,1312249
142,M00143,FY2025_26,2026-03-01,STORE,Acquisition,1984903


In [20]:
pd.DataFrame({
    'revenue_per_customer':summary['revenue']/summary['customers']*10000000,

    'marketing_spend_per_customer': summary['marketing_spend']/summary['customers']*10000000
}).round(4)

,revenue_per_customer,marketing_spend_per_customer
period_id,,
FY2024_25,100000.0,7000.0000
FY2025_26,80000.0,7323.0769


In [13]:
orders.groupby('period_id')['order_id'].nunique()/orders.groupby('period_id')['customer_id'].nunique()

,0
period_id,
FY2024_25,5.0
FY2025_26,4.0


In [11]:
orders.groupby('period_id')['net_revenue'].sum()/orders.groupby('period_id')['order_id'].nunique()

,0
period_id,
FY2024_25,20000.0
FY2025_26,20000.0
